# ATLAS Wind Atlas: scenario plotting

This notebook creates maps from the scenario wind dataset produced by the previous wind conversion notebook.

Expected input files are the final 8 variable NetCDF files produced by `5.2.2_ATLAS_wind_conversion_scenarios`:

```text
u10_corrected
v10_corrected
wds_corrected
dir_corrected
delta_u10
delta_v10
delta_wds
delta_dir
```

The notebook is written as a reusable plotting template. The user only needs to edit the configuration cells.

## Method overview

The notebook opens one scenario wind dataset for the continental area and, when available, one scenario wind dataset for the islands.

It then produces maps for selected variables. For Chile like domains, the continental area is plotted as the main map and the islands are shown as small inset maps.

The plotting logic is the same for all variables. Only the color scale, label and output file name change.

## 0. Load libraries

Run this cell first. It imports the plotting and geospatial libraries used throughout the notebook.

In [1]:
from pathlib import Path
import warnings

import cartopy.crs as ccrs
import cartopy.feature as cfeature
import geopandas as gpd
import matplotlib.pyplot as plt
import numpy as np
import rioxarray  # noqa: F401. Needed to activate the .rio accessor in xarray
import xarray as xr
from cartopy.mpl.gridliner import LONGITUDE_FORMATTER, LATITUDE_FORMATTER
from matplotlib.colors import LinearSegmentedColormap, Normalize, TwoSlopeNorm

warnings.filterwarnings("ignore")

## 1. User settings

Edit only this cell for a standard run.

The notebook expects the scenario conversion output from the previous step. By default it tries to plot both `continental` and `islands`. If the islands file does not exist, the notebook still produces the continental map.

In [2]:
# Country or study area name used in folder names and file names.
COUNTRY = "chile"

# CMIP6 model and scenario used in the previous scenario notebooks.
MODEL = "CNRM-ESM2-1"
EXPERIMENT = "ssp370"

# Future period used in the previous scenario notebooks.
START_YEAR = 2020
END_YEAR = 2050

# Month to plot when ANNUAL_PLOT is False.
# Example: MONTH = 1 means January.
#MONTH = 1

# If True, the notebook opens the 12 monthly files, computes the annual mean,
# and saves figures using "annual" in the output file name instead of "m<month>".
ANNUAL_PLOT = True

# Areas expected from the conversion notebook.
CONTINENTAL_AREA_NAME = "continental"
ISLANDS_AREA_NAME = "islands"

# If True, the notebook tries to add islands as inset maps.
PLOT_ISLANDS_INSETS = True

# Optional boundary shapefile. Set to None if no shapefile is available.
# Example: BOUNDARY_FILE = Path("../data/boundaries/chile_regions.shp")
BOUNDARY_FILE = None

# Output image resolution.
FIGURE_DPI = 150


## 2. Folder configuration

The paths are relative and anonymous, so the notebook can be shared without exposing local server folders.

Expected monthly input files:

```text
../data/atlas_data/<country>/<model>/<scenario>/10m_wind_corrected_<country>_m<month>_<scenario>_<area>_<start>_<end>.nc
```

If `ANNUAL_PLOT = True`, the notebook opens the 12 monthly files one by one and computes the annual mean before plotting.

Expected output:

```text
../figures/<country>/<model>/<scenario>/
```

Monthly figures are saved with `_m<month>_` in the file name. Annual figures are saved with `_annual_` in the file name.


In [3]:
BASE_DIR = Path("/mnt/DATA/PROGETTI/27_WMO_ATLAS/ATLAS")
FIGURE_BASE_DIR = Path("/mnt/DATA/PROGETTI/27_WMO_ATLAS/figures")
ATLAS_DIR = BASE_DIR / COUNTRY / "scaling" / EXPERIMENT / MODEL
OUTPUT_FIGURE_DIR = FIGURE_BASE_DIR / COUNTRY / EXPERIMENT / MODEL
OUTPUT_FIGURE_DIR.mkdir(parents=True, exist_ok=True)

In [3]:
BASE_DIR = Path("../data")
FIGURE_BASE_DIR = Path("../figures")

# Folder containing the scenario conversion outputs.
ATLAS_DIR = BASE_DIR / "atlas_data" / COUNTRY / MODEL / EXPERIMENT

# Folder where figures will be saved.
OUTPUT_FIGURE_DIR = FIGURE_BASE_DIR / COUNTRY / MODEL / EXPERIMENT
OUTPUT_FIGURE_DIR.mkdir(parents=True, exist_ok=True)

print("Input scenario folder:", ATLAS_DIR)
print("Output figure folder:", OUTPUT_FIGURE_DIR)


## 3. Variable configuration

This cell defines the variables that can be plotted. You can remove variables from `VARIABLES_TO_PLOT` if you only need a subset of maps.

In [4]:
VARIABLE_SETTINGS = {
    "u10_corrected": {
        "title": "Future corrected 10 m U wind component",
        "label": "m/s",
        "vmin": -8,
        "vmax": 8,
        "cmap": "RdBu_r",
        "norm": "centered",
        "file_prefix": "u10_corrected",
    },
    "v10_corrected": {
        "title": "Future corrected 10 m V wind component",
        "label": "m/s",
        "vmin": -8,
        "vmax": 8,
        "cmap": "RdBu_r",
        "norm": "centered",
        "file_prefix": "v10_corrected",
    },
    "wds_corrected": {
        "title": "Future corrected 10 m wind speed",
        "label": "m/s",
        "vmin": 0,
        "vmax": 5,
        "cmap": "BuPu",
        "norm": "linear",
        "file_prefix": "wds_corrected",
    },
    "dir_corrected": {
        "title": "Future corrected 10 m wind direction",
        "label": "degrees from North",
        "vmin": 0,
        "vmax": 360,
        "cmap": "twilight",
        "norm": "linear",
        "file_prefix": "dir_corrected",
    },
    "delta_u10": {
        "title": "Projected change in 10 m U wind component",
        "label": "m/s",
        "vmin": -10,
        "vmax": 10,
        "cmap": "RdBu_r",
        "norm": "centered",
        "file_prefix": "delta_u10",
    },
    "delta_v10": {
        "title": "Projected change in 10 m V wind component",
        "label": "m/s",
        "vmin": -10,
        "vmax": 10,
        "cmap": "RdBu_r",
        "norm": "centered",
        "file_prefix": "delta_v10",
    },
    "delta_wds_prc": {
        "title": "Projected change in 10 m wind speed \n % Anomaly",
        "label": "",
        "vmin": -10,
        "vmax": 10,
        "cmap": "bwr",
        "norm": "centered",
        "file_prefix": "delta_wds",
    },
    "delta_dir_prc": {
        "title": "Projected change in 10 m wind direction \n % Anomaly",
        "label": "",
        "vmin": -20,
        "vmax": 20,
        "cmap": "bwr",
        "norm": "centered",
        "file_prefix": "delta_dir",
    },
}

# Choose which variables to plot.
VARIABLES_TO_PLOT = [
    #"wds_corrected",
    #"dir_corrected",
    "delta_wds_prc",
    #"delta_dir",
]

# To plot all eight variables, uncomment the following line.
# VARIABLES_TO_PLOT = list(VARIABLE_SETTINGS)

## 4. Optional islands inset settings

These extents are specific to Chile. If the study area is different, either update the values or set `PLOT_ISLANDS_INSETS = False` in the user settings.

In [5]:
ISLAND_INSETS = [
    {
        "name": "Juan Fernandez",
        "extent": [-80.9, -78.7, -34.1, -33.5],
        "position": [0.08, 0.66, 0.32, 0.22],
    },
    {
        "name": "Desventuradas",
        "extent": [-80.2, -79.85, -26.36, -26.25],
        "position": [0.08, 0.40, 0.32, 0.22],
    },
    {
        "name": "Rapa Nui",
        "extent": [-109.6, -109.1, -27.4, -26.9],
        "position": [0.08, 0.14, 0.32, 0.22],
    },
]

## 5. File name helpers

If your previous notebooks use a different file naming convention, update only these functions.

In [6]:
def period_label():
    """Return the label used in output file names."""
    if ANNUAL_PLOT:
        return "annual"
    return f"m{MONTH}"


def scenario_wind_file(area_name, month=None):
    """Return one scenario wind file produced by the conversion notebook.

    Parameters
    ----------
    area_name : str
        Area identifier, for example "continental" or "islands".
    month : int or None
        Month number. If None, the global MONTH setting is used.
    """
    if month is None:
        month = MONTH

    file_name = (
        f"10m_wind_corrected_"
        f"{COUNTRY}_m{month}_{EXPERIMENT}_{area_name}_{START_YEAR}_{END_YEAR}.nc"
    )
    return ATLAS_DIR / file_name


def figure_file(variable_name):
    """Return the output figure path for one variable."""
    settings = VARIABLE_SETTINGS[variable_name]
    file_name = (
        f"{settings['file_prefix']}_{COUNTRY}_{period_label()}_"
        f"{EXPERIMENT}_{START_YEAR}_{END_YEAR}.png"
    )
    return OUTPUT_FIGURE_DIR / file_name


## 6. Open input data

This cell opens the continental dataset and, when available, the islands dataset.

If `ANNUAL_PLOT = False`, only the selected monthly file is opened.

If `ANNUAL_PLOT = True`, the notebook opens the twelve monthly files one by one, concatenates them along a temporary `month` dimension, and computes the annual mean. This annual dataset is then used by the plotting functions exactly like a monthly dataset.


In [7]:
DIRECTION_VARIABLES = ["dir_corrected", "delta_dir"]


def clean_spatial_metadata(ds):
    """Attach EPSG:4326 and remove auxiliary spatial reference variables."""
    ds = ds.rio.write_crs("EPSG:4326")
    ds = ds.drop_vars("spatial_ref", errors="ignore")
    return ds


def open_monthly_scenario_dataset(area_name, month, required=True):
    """Open one monthly scenario wind dataset."""
    path = scenario_wind_file(area_name, month=month)

    if not path.exists():
        message = f"Scenario file not found for area '{area_name}', month {month}: {path}"
        if required:
            raise FileNotFoundError(message)
        print(message)
        return None

    print(f"Opening {area_name} dataset for month {month}:", path)
    ds = xr.open_dataset(path, chunks={"latitude": 1000, "longitude": 1000})
    ds = clean_spatial_metadata(ds)
    return ds


def circular_monthly_mean_degrees(da, output_range="0_360"):
    """Compute a circular mean for variables expressed in degrees.

    This avoids artefacts around the 0/360 degree discontinuity when annual
    means are computed for wind direction variables.
    """
    radians = np.deg2rad(da)
    sin_mean = np.sin(radians).mean("month", skipna=True)
    cos_mean = np.cos(radians).mean("month", skipna=True)
    angle = np.rad2deg(np.arctan2(sin_mean, cos_mean))

    if output_range == "minus180_180":
        angle = ((angle + 180) % 360) - 180
    else:
        angle = angle % 360

    angle.attrs = da.attrs
    return angle


def compute_annual_mean(monthly_ds):
    """Compute the annual mean of a monthly scenario dataset.

    Linear variables are averaged with a standard arithmetic mean.
    Direction variables are averaged with a circular mean in degrees.
    """
    annual_vars = {}

    for variable_name, da in monthly_ds.data_vars.items():
        if variable_name == "dir_corrected":
            annual_vars[variable_name] = circular_monthly_mean_degrees(da, output_range="0_360")
        elif variable_name == "delta_dir":
            annual_vars[variable_name] = circular_monthly_mean_degrees(da, output_range="minus180_180")
        else:
            annual_vars[variable_name] = da.mean("month", skipna=True)

    annual_ds = xr.Dataset(annual_vars, coords={coord: monthly_ds.coords[coord] for coord in monthly_ds.coords if coord != "month"})
    annual_ds.attrs = monthly_ds.attrs
    return clean_spatial_metadata(annual_ds)


def open_annual_scenario_dataset(area_name, required=True):
    """Open the 12 monthly files for one area and return their annual mean."""
    monthly_datasets = []

    for month in range(1, 13):
        ds = open_monthly_scenario_dataset(area_name, month=month, required=required)
        if ds is None:
            return None
        ds = ds.expand_dims(month=[month])
        monthly_datasets.append(ds)

    print(f"Computing annual mean for area '{area_name}'...")
    monthly_stack = xr.concat(monthly_datasets, dim="month")
    annual_ds = compute_annual_mean(monthly_stack)

    for ds in monthly_datasets:
        ds.close()

    return annual_ds


def open_scenario_dataset(area_name, required=True):
    """Open either one monthly dataset or the annual mean dataset."""
    if ANNUAL_PLOT:
        return open_annual_scenario_dataset(area_name, required=required)
    return open_monthly_scenario_dataset(area_name, month=MONTH, required=required)


xdf_continent = open_scenario_dataset(CONTINENTAL_AREA_NAME, required=True)
xdf_islands = open_scenario_dataset(ISLANDS_AREA_NAME, required=False)

print("Continental variables:", list(xdf_continent.data_vars))
if xdf_islands is not None:
    print("Islands variables:", list(xdf_islands.data_vars))


Opening continental dataset for month 1: /mnt/DATA/PROGETTI/27_WMO_ATLAS/ATLAS/chile/scaling/ssp370/CNRM-ESM2-1/10m_wind_corrected_chile_m1_ssp370_continental_2020_2050.nc
Opening continental dataset for month 2: /mnt/DATA/PROGETTI/27_WMO_ATLAS/ATLAS/chile/scaling/ssp370/CNRM-ESM2-1/10m_wind_corrected_chile_m2_ssp370_continental_2020_2050.nc
Opening continental dataset for month 3: /mnt/DATA/PROGETTI/27_WMO_ATLAS/ATLAS/chile/scaling/ssp370/CNRM-ESM2-1/10m_wind_corrected_chile_m3_ssp370_continental_2020_2050.nc
Opening continental dataset for month 4: /mnt/DATA/PROGETTI/27_WMO_ATLAS/ATLAS/chile/scaling/ssp370/CNRM-ESM2-1/10m_wind_corrected_chile_m4_ssp370_continental_2020_2050.nc
Opening continental dataset for month 5: /mnt/DATA/PROGETTI/27_WMO_ATLAS/ATLAS/chile/scaling/ssp370/CNRM-ESM2-1/10m_wind_corrected_chile_m5_ssp370_continental_2020_2050.nc
Opening continental dataset for month 6: /mnt/DATA/PROGETTI/27_WMO_ATLAS/ATLAS/chile/scaling/ssp370/CNRM-ESM2-1/10m_wind_corrected_chile_m6_

### Opening present day climate data: used as reference to compute the % anomalies

In [9]:
if ANNUAL_PLOT:

    cont_files = xr.open_mfdataset(
        #"../data/atlas_data/chile/10m_wind_integrated_chile_m*_continental.nc",
        "/mnt/DATA/PROGETTI/27_WMO_ATLAS/ATLAS/chile/scaling/10m_wind_integrated_chile_m*_continental.nc",
        combine="nested",
        concat_dim="month",
        chunks={"latitude": 1000, "longitude": 1000}
    )

    isla_files = xr.open_mfdataset(
        #"../data/atlas_data/chile/10m_wind_integrated_chile_m*_islands.nc",
        "/mnt/DATA/PROGETTI/27_WMO_ATLAS/ATLAS/chile/scaling/10m_wind_integrated_chile_m*_islands.nc",
        combine="nested",
        concat_dim="month",
        chunks={"latitude": 1000, "longitude": 1000}
    )

    cont_ref_ds = cont_files.mean("month")
    isla_ref_ds = isla_files.mean("month")

else:

    cont_ref_ds = xr.open_dataset(
        #f"../data/atlas_data/chile/10m_wind_integrated_chile_m{month}_continental.nc"
        f"/mnt/DATA/PROGETTI/27_WMO_ATLAS/ATLAS/chile/scaling/10m_wind_integrated_chile_m{month}_continental.nc"
    )

    isla_ref_ds = xr.open_dataset(
        #f"../data/atlas_data/chile/10m_wind_integrated_chile_m{month}_islands.nc"
        f"/mnt/DATA/PROGETTI/27_WMO_ATLAS/ATLAS/chile/scaling/10m_wind_integrated_chile_m{month}_islands.nc"
    )

In [10]:
wds_cont_delta_pct = (
    xdf_continent["delta_wds"]
    / cont_ref_ds["wds_integrated"]
) * 100

wds_isla_delta_pct = (
    xdf_islands["delta_wds"]
    / isla_ref_ds["wds_integrated"]
) * 100
wds_cont_delta_pct = wds_cont_delta_pct.to_dataset(name='delta_wds_prc')
wds_isla_delta_pct     = wds_isla_delta_pct.to_dataset(name='delta_wds_prc')

## 7. Open optional boundaries

If a boundary shapefile is provided, it is plotted on top of the maps. Otherwise the notebook uses only Cartopy coastlines and borders.

In [11]:
def load_boundaries(boundary_file):
    """Load optional geographic boundaries."""
    if boundary_file is None:
        return None

    boundary_file = Path(boundary_file)
    if not boundary_file.exists():
        print("Boundary file not found. Continuing without boundaries:", boundary_file)
        return None

    boundaries = gpd.read_file(boundary_file)
    boundaries = boundaries.to_crs("EPSG:4326")
    return boundaries


boundaries = load_boundaries(BOUNDARY_FILE)

## 8. Plotting helper functions

These functions are reused for all maps.

In [12]:
def get_spatial_names(da):
    """Return latitude and longitude names used by a DataArray."""
    lat_candidates = ["latitude", "lat", "y"]
    lon_candidates = ["longitude", "lon", "x"]

    lat_name = next((name for name in lat_candidates if name in da.coords or name in da.dims), None)
    lon_name = next((name for name in lon_candidates if name in da.coords or name in da.dims), None)

    if lat_name is None or lon_name is None:
        raise ValueError(
            "Could not identify latitude and longitude coordinates. "
            f"Available coordinates are: {list(da.coords)}"
        )

    return lat_name, lon_name


def standardize_spatial_names(data):
    """Rename spatial coordinates to latitude and longitude when needed."""
    lat_name, lon_name = get_spatial_names(data)
    rename = {}
    if lat_name != "latitude":
        rename[lat_name] = "latitude"
    if lon_name != "longitude":
        rename[lon_name] = "longitude"
    if rename:
        data = data.rename(rename)
    return data


def cell_edges(coord):
    """Calculate cell edges from cell centres for pcolormesh."""
    coord = np.asarray(coord, dtype=float)
    if coord.size < 2:
        raise ValueError("At least two coordinate values are required to calculate cell edges.")

    d = np.diff(coord)
    edges = np.empty(coord.size + 1)
    edges[1:-1] = coord[:-1] + d / 2
    edges[0] = coord[0] - d[0] / 2
    edges[-1] = coord[-1] + d[-1] / 2
    return edges


def subset_by_extent(data, extent):
    """Subset a DataArray using an extent in lon_min, lon_max, lat_min, lat_max order."""
    xmin, xmax, ymin, ymax = extent
    data = standardize_spatial_names(data)

    lat = data.latitude
    if lat[0] > lat[-1]:
        lat_slice = slice(ymax, ymin)
    else:
        lat_slice = slice(ymin, ymax)

    return data.sel(longitude=slice(xmin, xmax), latitude=lat_slice)


def make_norm(settings):
    """Create the color normalization object for one variable."""
    vmin = settings.get("vmin")
    vmax = settings.get("vmax")

    if settings.get("norm") == "centered":
        return TwoSlopeNorm(vmin=vmin, vcenter=0, vmax=vmax)

    return Normalize(vmin=vmin, vmax=vmax)


def prepare_dataarray(ds, variable_name):
    """Extract one variable, standardize coordinates and sort the grid."""
    if variable_name not in ds.data_vars:
        raise KeyError(
            f"Variable '{variable_name}' was not found. "
            f"Available variables are: {list(ds.data_vars)}"
        )

    data = standardize_spatial_names(ds[variable_name])
    data = data.sortby("latitude").sortby("longitude")
    data = data.rio.set_spatial_dims(x_dim="longitude", y_dim="latitude")
    data = data.rio.write_crs("EPSG:4326")
    return data

## 9. Map drawing functions

The main map shows the continental area. Islands are added as insets when the islands dataset exists and the inset option is active.

In [13]:
def add_base_layers(ax):
    """Add simple geographic layers to a Cartopy axis."""
    ax.add_feature(cfeature.OCEAN, facecolor="lightblue")
    ax.add_feature(cfeature.LAKES, edgecolor="black", facecolor="lightblue")
    ax.add_feature(cfeature.RIVERS, edgecolor="lightblue")
    ax.add_feature(cfeature.COASTLINE, linewidth=1.0)
    ax.add_feature(cfeature.BORDERS, linewidth=0.8)


def add_gridlines(ax, label_size=10):
    """Add formatted longitude and latitude gridlines."""
    gl = ax.gridlines(draw_labels=True, linewidth=0.4, color="gray", alpha=0.5, linestyle="--")
    gl.top_labels = False
    gl.right_labels = False
    gl.xformatter = LONGITUDE_FORMATTER
    gl.yformatter = LATITUDE_FORMATTER
    gl.xlabel_style = {"size": 20}
    gl.ylabel_style = {"size": 20}
    return gl


def plot_boundaries(ax):
    """Plot optional study area boundaries."""
    if boundaries is not None:
        boundaries.boundary.plot(
            ax=ax,
            edgecolor="black",
            linewidth=0.6,
            transform=ccrs.PlateCarree(),
        )


def plot_data_on_axis(ax, data, cmap, norm):
    """Plot one gridded DataArray on a Cartopy axis."""
    data = data.sortby("latitude").sortby("longitude")

    lon = data.longitude.values
    lat = data.latitude.values
    values = np.ma.masked_invalid(data.values)

    lon_edges = cell_edges(lon)
    lat_edges = cell_edges(lat)
    lon2d, lat2d = np.meshgrid(lon_edges, lat_edges)

    image = ax.pcolormesh(
        lon2d,
        lat2d,
        values,
        transform=ccrs.PlateCarree(),
        cmap=cmap,
        norm=norm,
        shading="flat",
    )
    return image


def plot_inset(ax, data, title, extent, cmap, norm):
    """Plot one islands inset."""
    subset = subset_by_extent(data, extent)
    plot_data_on_axis(ax, subset, cmap, norm)
    plot_boundaries(ax)
    ax.set_extent(extent, crs=ccrs.PlateCarree())
    ax.set_title(title, fontsize=24, pad=2)
    add_gridlines(ax, label_size=10)


def draw_map(variable_name, continent_ds, islands_ds=None, save=True):
    """Draw and save one scenario map."""
    settings = VARIABLE_SETTINGS[variable_name]

    continent_data = prepare_dataarray(continent_ds, variable_name)
    islands_data = prepare_dataarray(islands_ds, variable_name) if islands_ds is not None else None

    cmap = settings["cmap"]
    norm = make_norm(settings)

    fig = plt.figure(figsize=(10, 14))
    ax = fig.add_subplot(111, projection=ccrs.PlateCarree())

    add_base_layers(ax)
    image = plot_data_on_axis(ax, continent_data, cmap, norm)
    plot_boundaries(ax)

    lon = continent_data.longitude.values
    lat = continent_data.latitude.values
    ax.set_extent([lon.min() - 1, lon.max() + 1, lat.min() - 1, lat.max() + 1], crs=ccrs.PlateCarree())

    add_gridlines(ax, label_size=10)
    ax.set_title(settings["title"], fontsize=24)

    colorbar = fig.colorbar(image, ax=ax, shrink=0.75, pad=0.03, aspect=30)
    colorbar.set_label(settings["label"], fontsize=20)
    colorbar.ax.tick_params(labelsize=20)
    if PLOT_ISLANDS_INSETS and islands_data is not None:
        fig.subplots_adjust(left=0.42, right=0.88)
        for inset in ISLAND_INSETS:
            ax_inset = fig.add_axes(inset["position"], projection=ccrs.PlateCarree())
            plot_inset(
                ax_inset,
                islands_data,
                title=inset["name"],
                extent=inset["extent"],
                cmap=cmap,
                norm=norm,
            )
    else:
        fig.subplots_adjust(left=0.08, right=0.92)

    output_path = figure_file(variable_name)
    if save:
        fig.savefig(output_path, dpi=FIGURE_DPI, bbox_inches="tight")
        print("Saved figure:", output_path)
        plt.close(fig)

    return fig

## 10. Create maps

This cell creates one PNG file for each variable listed in `VARIABLES_TO_PLOT`.

In [ ]:
generated_figures = []

for variable_name in VARIABLES_TO_PLOT:
    print("=" * 80)
    print("Plotting variable:", variable_name)
    if "prc" in variable_name:
        draw_map(variable_name, wds_cont_delta_pct, wds_isla_delta_pct, save=True)
        generated_figures.append(figure_file(variable_name))
    else:
        draw_map(variable_name, xdf_continent, xdf_islands, save=True)
        generated_figures.append(figure_file(variable_name))

print("=" * 80)
print("Scenario plotting completed.")
print("Generated figures:")
for path in generated_figures:
    print(path)

Plotting variable: delta_wds_prc


## 11. Optional quick preview

Use this cell to display one map inside the notebook without saving it again.

In [17]:
#Example preview. Change the variable name if needed.
fig = draw_map("wds_corrected", xdf_continent, xdf_islands, save=True)
plt.show()

IOStream.flush timed out


Saved figure: /mnt/DATA/PROGETTI/27_WMO_ATLAS/figures/chile/ssp370/CNRM-ESM2-1/wds_corrected_chile_annual_ssp370_2020_2050.png
